In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 33.5 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [ ]:
from datasets import load_dataset

dataset = load_dataset("tatsu-lab/alpaca", split="train[:2000]")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [ ]:
dataset["input"][0] != ""

False

In [ ]:
def format_prompt(example):

    if example["input"] != "":
        text = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    else:
        text = f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""

    return {"text": text}

dataset = dataset.map(format_prompt)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="alpaca_qwen",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=20,
    save_steps=200,
    fp16=True,
    report_to="none"
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,4.365691
40,0.630881
60,0.531164
80,0.515739
100,0.424297
120,0.584547
140,0.406561
160,0.520598
180,0.427567
200,0.484770


TrainOutput(global_step=500, training_loss=0.6326701259613037, metrics={'train_runtime': 222.7642, 'train_samples_per_second': 8.978, 'train_steps_per_second': 2.245, 'total_flos': 1101123944448000.0, 'train_loss': 0.6326701259613037})

In [ ]:
trainer.model.save_pretrained("alpaca_qwen_lora")
tokenizer.save_pretrained("alpaca_qwen_lora")

('alpaca_qwen_lora/tokenizer_config.json',
 'alpaca_qwen_lora/chat_template.jinja',
 'alpaca_qwen_lora/tokenizer.json')

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

prompt = """
### Instruction:
Explain machine learning simply

### Response:
"""

print(pipe(prompt, max_new_tokens=100)[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



### Instruction:
Explain machine learning simply

### Response:
Machine learning is a type of artificial intelligence that allows computers to learn from data and improve their performance over time. It involves training algorithms on large datasets in order to make predictions or decisions based on new information. Machine learning can be used for tasks such as image recognition, natural language processing, and fraud detection.


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
trainer.model.push_to_hub("Sanjarbek1024/alpaca-qwen-0.5b-lora")
tokenizer.push_to_hub("Sanjarbek1024/alpaca-qwen-0.5b-lora")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  24%|##4       |  529kB / 2.18MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpaz619j_v/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/Sanjarbek1024/alpaca-qwen-0.5b-lora/commit/68bfe4b00d560fef5fdaafdd70d2df733669ec80', commit_message='Upload tokenizer', commit_description='', oid='68bfe4b00d560fef5fdaafdd70d2df733669ec80', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sanjarbek1024/alpaca-qwen-0.5b-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='Sanjarbek1024/alpaca-qwen-0.5b-lora'), pr_revision=None, pr_num=None)

In [ ]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id="Sanjarbek1024/alpaca-qwen-0.5b-lora",
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/Sanjarbek1024/alpaca-qwen-0.5b-lora/commit/ea4de90a23e9303017b9289d00ae445887b4a18b', commit_message='Upload README.md with huggingface_hub', commit_description='', oid='ea4de90a23e9303017b9289d00ae445887b4a18b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Sanjarbek1024/alpaca-qwen-0.5b-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='Sanjarbek1024/alpaca-qwen-0.5b-lora'), pr_revision=None, pr_num=None)